# A3：前车急刹 —— AEB 紧急制动 Python 闭环
# A3: Lead-Vehicle Emergency Braking — AEB Closed-Loop Test

## 一、实验原理 | Principle

### 1.1 AEB（自动紧急制动）
**AEB (Autonomous Emergency Braking)** 是 L2 级辅助驾驶的核心安全功能：
- 传感器持续监测前方目标，计算 **TTC（碰撞时间）**
- TTC 低于阈值时自动制动，无需驾驶员介入
- Euro NCAP 要求：50km/h 下对静止目标必须避免碰撞或大幅减速

### 1.2 分级制动策略 | Staged Braking
真实 AEB 不是"一脚刹死"，而是分级介入（兼顾安全与舒适）：
| 阶段 | 增强TTC 条件 | 制动强度 | 目的 |
|------|-------------|---------|------|
| 预警 | TTC < 4.0s | 轻制动 -4.0 m/s² | 提醒驾驶员，温和减速 |
| 部分制动 | TTC < 2.5s | 中制动 -6.0 m/s² | 大幅降低车速 |
| 全力制动 | TTC < 1.8s | 全制动 -8.0 m/s² | 最后避免碰撞的机会 |

**增强 TTC**：朴素 TTC = 相对距离/相对速度，假设前车匀速。
前车急刹时真实 TTC 更短。本实验估计前车加速度，解二次方程
`rel_d = rel_v·t + ½·rel_a·t²` 求更准确的 TTC——这是本实验的核心算法改进。

**迟滞（Hysteresis）**：制动一旦触发，不立即释放，直到 TTC 回升到阈值+0.3s，
防止制动在阈值附近频繁抖动。

### 1.3 误触发率 | False Trigger Rate
AEB 的最大工程挑战不是"刹不住"，而是"不该刹的时候刹了"：
- 高速上误触发 → 后车追尾风险
- 评价方法：在**安全场景**（前车远离/匀速）中统计 AEB 激活比例
- 行业要求：误触发率 < 10%（本实验的通过线）

### 1.4 实验设计
- **危险场景**：Target 在 t=3s 以 4m/s² 急刹到 5m/s → 检验是否碰撞、碰撞相对速度
- **安全场景**：Ego 比 Target 慢，间距拉大 → 统计误触发率
- 两个场景都用 **真实 esmini 引擎**（ctypes 调用 `libesminiLib.so`）

## 二、实验目的 | Objectives
1. 实现完整的三级 AEB 制动逻辑（含迟滞）
2. 理解"避免碰撞"与"避免误触发"的权衡（灵敏度 vs 特异度）
3. 掌握 `BaseController` 接口在紧急场景下的应用
4. 学会用**碰撞相对速度**量化 AEB 的减伤效果

## 三、学生任务 | Your Tasks
1. **阅读** `FullAEBController`，理解分级阈值和迟滞逻辑
2. **运行**危险场景，确认无碰撞，记录最小 TTC
3. **运行**安全场景，计算误触发率（应 < 10%）
4. **调参**：把全力制动阈值从 1.2 改为 0.8 —— 碰撞了吗？误触发率变了吗？
5. **思考题**：为什么不能只用一级制动（TTC<1.5 直接 -8）？从舒适性和误触发两个角度分析

## 四、验证内容 | Verification
- [ ] esmini 成功初始化危险场景和安全场景
- [ ] 危险场景：`assert not danger_collision` 通过（无碰撞）
- [ ] 安全场景：`assert false_trigger_rate < 0.10` 通过（误触发率 < 10%）
- [ ] 碰撞相对速度（如发生）被正确记录
- [ ] 速度曲线显示三级制动依次介入

## 五、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

| 指标 | 通过线 | 物理意义 |
|------|--------|---------|
| `danger_collision` | 必须为 False | 危险场景（前车急刹）下是否碰撞——AEB 的核心使命 |
| `false_trigger_rate` | < 0.10 | 安全场景下 AEB 误激活的比例。**这是 AEB 最难的指标**：刹得住容易，不该刹时不刹难 |
| 碰撞相对速度 | 越小越好（如发生） | 即使没完全避免碰撞，把相对速度从 30km/h 降到 10km/h 也能大幅降低伤亡率（Euro NCAP 评分项） |

**False trigger rate** is the Achilles' heel of AEB. A system that brakes at every shadow will avoid all collisions — and cause rear-end crashes from following vehicles. The 10% bar is lenient for teaching; production systems target < 1%.

**三级制动的工程意义**：真实 AEB 不是"一脚刹死"，而是分级介入——
- TTC < 4.0s：轻制动 -4.0 m/s²（提醒驾驶员，同时温和减速）
- TTC < 2.5s：中制动 -6.0 m/s²（大幅降低车速）
- TTC < 1.8s：全制动 -8.0 m/s²（最后机会）
分级的好处：误触发时只是轻点刹车，不会造成追尾；同时给驾驶员接管的机会。

### 5.2 增强 TTC vs 朴素 TTC | Enhanced vs Naive TTC

本实验的核心算法改进：

**朴素 TTC** = 相对距离 / 相对速度（假设前车匀速）
**增强 TTC**：估计前车加速度 $a_{lead}$，解二次方程 $d = v·t + ½·a·t²$ 求碰撞时间

| 方法 | 前车急刹时的表现 | 教学结论 |
|------|---------------|---------|
| 朴素 TTC | **高估** TTC（以为前车还在匀速），制动过晚 → 碰撞 | 挑战任务1让学生亲眼看到碰撞 |
| 增强 TTC | 正确估计更短的 TTC，提前制动 → 避免碰撞 | 前车加速度估计是 AEB 的核心 |

**低通滤波的作用**：前车加速度用差分估计，噪声大。直接用原始差分会导致 TTC 剧烈抖动 → 制动抖动。滤波是在"响应速度"和"稳定性"之间的权衡。

### 5.3 结果合理性判断 | Is the Result Reasonable?

- **危险场景无碰撞 + 安全场景误触发率 < 5%**：优秀。分级阈值和迟滞整定得当。
- **危险场景无碰撞但误触发率 5~10%**：可接受，但偏保守。可以尝试把 4.0s 预警阈值收紧到 3.5s。
- **危险场景发生碰撞**：检查增强 TTC 是否正确实现。常见错误：加速度估计符号搞反，或二次方程求根取了错误的根。
- **误触发率 > 10%**：过于灵敏。检查：① 阈值是否太小；② 迟滞是否生效（`TTC > threshold + 0.3` 才释放）；③ 安全场景的前车是否真的在远离。

### 5.4 迟滞（Hysteresis）的作用 | Why Hysteresis?

没有迟滞时：TTC 在 1.8s 附近波动 → 制动在 -8 和 0 之间疯狂切换 → 乘客被晃吐 + 制动系统过热。
有迟滞时：TTC < 1.8 触发全力制动，直到 TTC 回升到 2.1 才释放 → 制动平稳。

这是控制工程的通用技巧：**继电器的回差、恒温器的死区，都是迟滞**。

### 5.5 误差来源分析 | Error Sources

1. **前车加速度估计延迟**：低通滤波引入约 0.2~0.3s 延迟。在 4m/s² 急刹下，0.3s 延迟意味着 Ego 多走了约 4m。这是增强 TTC 的固有代价。
2. **目标车冻结**：同 A1/A2，Target 的急刹动作在 ctypes 模式下可能不生效。本实验的"急刹"实际由 Python 端的目标车运动模型驱动（见代码中的 target 运动逻辑），esmini 只负责场景加载。
3. **轮胎-路面模型**：-8.0 m/s² 假设干路面（μ≈0.8）。湿滑路面（μ≈0.4）下最大减速度仅 -4 m/s²，三级阈值需要整体前移。

### 5.6 与理论预期的对比 | vs. Theory

| 理论/行业惯例 | 本实验 | 对比结论 |
|-------------|--------|---------|
| Euro NCAP：50km/h 对静止目标避免碰撞 | 危险场景相对速度约 30km/h | 场景难度低于 NCAP，适合教学入门 |
| 量产 AEB 误触发率 < 1% | 通过线 10% | 教学放宽 10 倍；让学生理解"从 10% 到 1%"是量产化的真正壁垒 |
| 增强 TTC 是 Mobileye/博世的核心专利方向 | 本实验复现了基本思想 | 算法思想与工业界一致，参数是简化版 |


> ⚠️ **架构诚实声明**：本实验采用混合架构——Target车辆由esmini引擎加载，但实测发现esmini在ctypes模式下目标车的脚本动作不生效（车辆冻结不动，等价于静态障碍物）；Ego车辆由Python运动学模型（Kinematic Bicycle Model）积分推进，通过`SE_ReportObjectPos`上报给esmini。这不是完整的esmini双车闭环，而是"esmini场景加载 + Python Ego闭环"的教学简化。如需全动态双车，请使用esmini二进制headless模式。


> ⚠️ **Architecture Disclosure**: This experiment uses a hybrid architecture — the Target vehicle is loaded by the esmini engine, but testing revealed that scripted target actions do not take effect in ctypes mode (vehicle frozen, equivalent to a static obstacle); the Ego vehicle is integrated by a Python kinematic bicycle model and reported to esmini via `SE_ReportObjectPos`. This is a teaching simplification ("esmini scene loading + Python Ego closed-loop"), not a full esmini two-vehicle closed loop. For fully dynamic two-vehicle simulation, use the esmini binary in headless mode.



In [ ]:
import sys
sys.dont_write_bytecode = True  # Disable .pyc cache
# Clean stale .pyc cache (esmini ctypes is sensitive to it)
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
# Force reload (bypass ctypes deadlock caused by stale .pyc)
import common.esmini_env
from common.esmini_env import EsminiEnv
print('OK: labs/common imported')

In [ ]:
# 从车辆库加载参数（链1：车型设计→智能驾驶）
from common.vehicle_library import load_vehicle
veh_params = load_vehicle('bev_sedan')
print(f"车辆: {veh_params['name']}, 质量{veh_params['mass_kg']}kg")
# 运动学参数用于Ego车辆模型
L_F = veh_params['kinematic']['l_f']
L_R = veh_params['kinematic']['l_r']
MAX_STEER = veh_params['kinematic']['max_steer_deg']
MAX_ACCEL = veh_params['kinematic']['max_accel_mss']


### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `ModuleNotFoundError: common` | 找不到 labs 库 | 检查 `sys.path` 是否包含 `/home/jovyan/labs` |
| `SE_InitWithArgs failed: -1` | esmini 初始化失败 | 检查场景文件路径；确认在场景目录运行 |
| `FileNotFoundError: aeb_brake.xosc` | 场景文件不存在 | 运行下面的场景生成 cell |
| `AssertionError: danger_collision` | 危险场景发生碰撞 | 降低制动阈值或增大制动力度 |
| `AssertionError: false_trigger_rate` | 误触发率超标 | 提高触发阈值，或检查迟滞逻辑 |
| 仿真卡死无响应 | ctypes 死锁（stale .pyc） | 重启 Kernel 后按顺序运行所有 cell |


## 1. 实现三级 AEB 控制器 | Implement Staged AEB Controller
继承 `BaseController`，实现带迟滞的三级制动。
Inherit `BaseController` and implement hysteresis-based staged braking.

**接口 | Interface:**
- 输入 `observation`：`{'ego': {'x','y','yaw','v','a'}, 'targets': [...], 'sim_time': t, 'dt': dt}`
- 输出 `action`：`{'target_a': 加速度, 'steer_angle': 转向角}`
- 本实验只用纵向控制，`steer_angle` 恒为 0


In [ ]:
class FullAEBController(BaseController):
    """Three-stage AEB with enhanced TTC (accounts for lead deceleration).

    Key insight: naive TTC = rel_dist / rel_v assumes lead vehicle moves
    at constant speed. When lead brakes hard, true TTC is SHORTER.
    We estimate lead acceleration and solve the quadratic equation:
        rel_d = rel_v * t + 0.5 * rel_a * t^2
    """

    def __init__(self, config=None):
        super().__init__(config)
        # Enhanced-TTC thresholds (seconds)
        self.t_warn = self.config.get('t_warn', 4.0)    # light braking
        self.t_part = self.config.get('t_part', 2.5)   # partial braking
        self.t_full = self.config.get('t_full', 1.8)   # full braking
        # Hysteresis: release only when TTC recovers above threshold + margin
        self.hyst = self.config.get('hysteresis', 0.3)
        self._active_level = 0  # 0=none, 1=warn, 2=partial, 3=full
        self.activations = 0    # brake activations (for false-trigger stats)
        self._prev_lead_v = None
        self._lead_a_est = 0.0

    def reset(self):
        self._active_level = 0
        self.activations = 0
        self._prev_lead_v = None
        self._lead_a_est = 0.0

    def _compute_ttc(self, ego, lead, dt):
        """Enhanced TTC accounting for lead vehicle acceleration."""
        rel_dist = abs(lead['x'] - ego['x']) - 4.5  # minus vehicle length
        rel_v = ego['v'] - lead['v']
        # Estimate lead acceleration (low-pass filtered)
        if self._prev_lead_v is not None:
            lead_a = (lead['v'] - self._prev_lead_v) / dt
            self._lead_a_est = 0.7 * self._lead_a_est + 0.3 * lead_a
        self._prev_lead_v = lead['v']
        # Relative acceleration (positive = gap closing faster)
        rel_a = -self._lead_a_est
        if rel_v <= 0.1 and rel_a <= 0.1:
            return float('inf'), rel_dist
        if rel_a > 0.1:
            # Solve 0.5*rel_a*t^2 + rel_v*t - rel_d = 0
            disc = rel_v ** 2 + 2 * rel_a * rel_dist
            if disc <= 0:
                return float('inf'), rel_dist
            return (-rel_v + disc ** 0.5) / rel_a, rel_dist
        return (rel_dist / rel_v if rel_v > 0.1 else float('inf')), rel_dist

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        dt = observation.get('dt', 0.05)
        if not targets:
            return {'target_a': 0.0, 'steer_angle': 0.0}
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        ttc, _ = self._compute_ttc(ego, lead, dt)

        # Staged braking with hysteresis on release
        if ttc < self.t_full:
            level = 3
        elif ttc < self.t_part:
            level = 2
        elif ttc < self.t_warn:
            level = 1
        else:
            if self._active_level == 3 and ttc < self.t_full + self.hyst:
                level = 3
            elif self._active_level == 2 and ttc < self.t_part + self.hyst:
                level = 2
            elif self._active_level == 1 and ttc < self.t_warn + self.hyst:
                level = 1
            else:
                level = 0

        # Count new activations (for false-trigger measurement)
        if level > 0 and self._active_level == 0:
            self.activations += 1
        self._active_level = level

        a = {0: 0.0, 1: -4.0, 2: -6.0, 3: -8.0}[level]
        return {'target_a': a, 'steer_angle': 0.0}

print('OK: FullAEBController defined')

## 2. 危险场景测试 | Danger Scenario Test
**真实 esmini 引擎**：Target 在 t=3s 以 4m/s² 急刹到 5m/s。
Ego 以 16m/s 跟进，AEB 必须避免碰撞。


In [ ]:
# Locate the A3 scenario files (new layout with fallback)
import os
candidates = [
    '/home/jovyan/labs/scenarios/A_esmini/A3_aeb/aeb_brake.xosc',
    '/home/jovyan/labs/ad2026/aeb_brake.xosc',
]
scenario_path = next((p for p in candidates if os.path.exists(p)), None)
assert scenario_path is not None, 'Scenario file not found! Check labs/scenarios/A_esmini/A3_aeb/'
print(f'OK: scenario = {scenario_path}')

In [ ]:
# Run danger scenario with real esmini closed-loop
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import EsminiEnv

controller = FullAEBController()
env = EsminiEnv(scenario=scenario_path, dt=0.05)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=16.0)
controller.reset()

traj, danger_collision, collision_rel_v = [], False, None
min_ttc, min_gap = float('inf'), float('inf')

for step in range(400):
    action = controller.step(obs)
    obs, done, info = env.step(action)
    t = obs['sim_time']
    ego = obs['ego']
    traj.append((t, ego['x'], ego['v'], action['target_a']))
    for tgt in obs.get('targets', []):
        rel_d = abs(tgt['x'] - ego['x']) - 4.5
        min_gap = min(min_gap, rel_d)
        rel_v = ego['v'] - tgt['v']
        if rel_v > 0.1:
            min_ttc = min(min_ttc, rel_d / rel_v)
        if rel_d < 0.5 and not danger_collision:
            danger_collision = True
            collision_rel_v = rel_v  # record impact speed
    if done or danger_collision:
        break

env.close()
print(f'OK: danger test done in {len(traj)} steps')
print(f'  collision={danger_collision}, min_ttc={min_ttc:.2f}s, min_gap={min_gap:.1f}m')
if danger_collision:
    print(f'  collision relative speed={collision_rel_v:.1f} m/s')

## 3. 安全场景误触发测试 | False-Trigger Test
**安全场景**：Ego 以 12m/s 行驶，比 Target（14m/s）慢，间距持续拉大。
AEB **不应该**制动。统计制动激活次数，计算误触发率。


In [ ]:
# Run safe scenario: ego slower than target, gap grows -> AEB must stay silent
controller2 = FullAEBController()
env2 = EsminiEnv(scenario=scenario_path, dt=0.05)
# Ego slower (12 m/s); target cruises at 14 m/s -> no closing, TTC = inf
obs2 = env2.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=12.0)
controller2.reset()

safe_steps, brake_steps = 0, 0
for step in range(200):
    action = controller2.step(obs2)
    obs2, done, info = env2.step(action)
    safe_steps += 1
    if abs(action['target_a']) > 0.5:
        brake_steps += 1
    if done:
        break

env2.close()
false_trigger_rate = brake_steps / safe_steps
print(f'OK: safe test done in {safe_steps} steps')
print(f'  brake active steps={brake_steps}, false_trigger_rate={false_trigger_rate:.3f}')
print(f'  AEB activations={controller2.activations}')

## 4. 评测断言 | Evaluation
- `danger_collision`：危险场景是否碰撞（必须为 False）
- `false_trigger_rate`：安全场景误触发率（必须 < 10%）
- `collision_rel_v`：如发生碰撞，记录碰撞相对速度（减伤效果量化）


In [ ]:
# Pass criteria: no collision in danger test AND false-trigger rate < 10%
assert not danger_collision, 'Danger scenario: collision occurred!'
assert false_trigger_rate < 0.10, f'False trigger rate {false_trigger_rate:.3f} >= 0.10!'
print('PASS: all assertions passed')
print(f'  min_ttc={min_ttc:.2f}s  danger_no_collision=True')
print(f'  false_trigger_rate={false_trigger_rate:.3f} < 0.10')

## 六、真实仿真轨迹可视化 | Visualization
下图是**真实仿真数据**（不是示意动画）：
- 左图：速度曲线（可见三级制动介入时机）
- 右图：减速度曲线（可见 -2.5 / -5.0 / -8.0 三档）


In [ ]:
import matplotlib.pyplot as plt

times = [t for t, x, v, a in traj]
speeds = [v for t, x, v, a in traj]
accels = [a for t, x, v, a in traj]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(times, speeds)
plt.xlabel('Time (s)'); plt.ylabel('Ego speed (m/s)')
plt.title('AEB Speed Profile (Real Sim)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(times, accels)
plt.xlabel('Time (s)'); plt.ylabel('Commanded accel (m/s^2)')
plt.title('AEB Braking Stages (Real Sim)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 七、🏆 挑战任务 | Challenges

1. **对比挑战**：把 `_compute_ttc` 改为朴素版本（`ttc = rel_dist / rel_v`，不考虑前车加速度）。
   - 危险场景还通过吗？（提示：大概率碰撞——这就是为什么需要增强 TTC）
   - 这证明了：**前车加速度估计是 AEB 的核心**

2. **调参挑战**：把 `t_full` 从 1.8 改为 1.2（更晚才全力制动）。
   - 碰撞了吗？`collision_rel_v` 是多少？
   - 误触发率变了吗？（灵敏度 vs 特异度的权衡）

3. **场景挑战**：把 Target 的制动强度从 4m/s² 改为 6m/s²（修改 `gen_aeb.py` 重新生成）。
   - 你的三级 AEB 还能避免碰撞吗？增强 TTC 提前了多少秒介入？


---

## 下一个实验 | Next →

**[A4_intersection_yield](./A4_intersection_yield.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
